# Canadian Parliament Fact-Check Llama Fine-Tuning Notebook

This notebook is a clean Colab-first training pipeline for your project.

It is built for **two modes**:
- **`TRAIN_MODE = "lora"`** → practical default
- **`TRAIN_MODE = "full"`** → optional full fine-tuning for stronger hardware

It uses:
- **Google Drive** for persistence
- your existing **`training_pairs.json`** prompt/completion dataset
- **TRL + Transformers + PEFT**
- a final **merged Hugging Face model**

The notebook is written to be readable first, fast second.

## 1) Install dependencies

This cell installs the training stack used in the rest of the notebook.
Run it once at the top of a fresh Colab session.

In [1]:
# Core fine-tuning stack.
!pip -q install -U     "transformers>=4.57.0"     "datasets>=3.0.0"     "trl>=0.17.0"     "peft>=0.17.0"     "accelerate>=1.4.0"     "bitsandbytes>=0.45.0"     "sentencepiece>=0.2.0"     "scikit-learn>=1.5.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 155.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.5/527.5 kB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 43.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 159.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 56.6 MB/s eta 0:00:00


## 2) Mount Google Drive and define project paths

All important artifacts are saved into Drive so you do not lose them when the Colab runtime resets.

In [2]:
from pathlib import Path
import shutil

from google.colab import drive
drive.mount("/content/drive")

# Change this only if your Drive folder structure is different.
PROJECT_ROOT = Path("/content/drive/MyDrive/AIGColab/NLP/FinalProject")

DATA_DIR = PROJECT_ROOT / "data"
RUNS_DIR = PROJECT_ROOT / "runs"
MODELS_DIR = PROJECT_ROOT / "models"
REPORTS_DIR = PROJECT_ROOT / "reports"

for folder in [DATA_DIR, RUNS_DIR, MODELS_DIR, REPORTS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# Preferred persistent dataset location in Drive.
DATA_PATH = DATA_DIR / "training_pairs.json"

# Optional convenience:
# If you manually uploaded training_pairs.json into /content in this Colab session,
# copy it into Drive automatically the first time.
uploaded_copy = Path("/content/training_pairs.json")
if not DATA_PATH.exists() and uploaded_copy.exists():
    shutil.copy2(uploaded_copy, DATA_PATH)
    print(f"Copied uploaded dataset to Drive: {DATA_PATH}")

print("PROJECT_ROOT =", PROJECT_ROOT)
print("DATA_PATH     =", DATA_PATH)

Mounted at /content/drive
PROJECT_ROOT = /content/drive/MyDrive/AIGColab/NLP/FinalProject
DATA_PATH     = /content/drive/MyDrive/AIGColab/NLP/FinalProject/data/training_pairs.json


## 3) Authenticate with Hugging Face and inspect hardware

For gated Meta Llama models, store your Hugging Face token in **Colab Secrets** as `HF_TOKEN`.

In [4]:
import torch

from google.colab import userdata
from huggingface_hub import login

HF_TOKEN = userdata.get("HF_TOKEN")
if not HF_TOKEN:
    raise ValueError(
        "Missing HF_TOKEN in Colab Secrets. Add it in Colab and rerun this cell."
    )

login(token=HF_TOKEN, add_to_git_credential=False)

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print("GPU:", gpu_name)
    print(f"GPU memory: {total_gb:.2f} GB")

CUDA available: True
GPU: NVIDIA A100-SXM4-80GB
GPU memory: 79.25 GB


## 4) Global configuration

This is the only place you should edit first.

The notebook is intentionally parameterized so you can keep one clean notebook and switch between LoRA and full fine-tuning.

In [5]:
import json
from datetime import datetime

# =========================
# Main training switches
# =========================
TRAIN_MODE = "lora"   # "lora" or "full"
BASE_MODEL = "meta-llama/Llama-3.1-8B-Instruct"

# =========================
# Reproducibility
# =========================
SEED = 3407

# =========================
# Data split
# =========================
VAL_SIZE = 0.10
TEST_SIZE = 0.10

# =========================
# Sequence length
# =========================
# We will auto-estimate a good value from the dataset later,
# then clamp it by this cap.
MAX_LENGTH_CAP = 2048

# =========================
# LoRA config
# =========================
LORA_R = 32
LORA_ALPHA = 32
LORA_DROPOUT = 0.0
LORA_TARGET_MODULES = [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    "gate_proj",
    "up_proj",
    "down_proj",
]

# =========================
# Optimization
# =========================
PER_DEVICE_TRAIN_BATCH_SIZE = 1
PER_DEVICE_EVAL_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8

NUM_EPOCHS_LORA = 4
NUM_EPOCHS_FULL = 3

LEARNING_RATE_LORA = 2e-4
LEARNING_RATE_FULL = 1e-5

WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.05
LR_SCHEDULER_TYPE = "cosine"
MAX_GRAD_NORM = 1.0

# =========================
# Logging / checkpointing
# =========================
LOGGING_STEPS = 10
EVAL_STEPS = 50
SAVE_STEPS = 50
SAVE_TOTAL_LIMIT = 2
EARLY_STOPPING_PATIENCE = 2

# =========================
# Generation evaluation
# =========================
EVAL_MAX_NEW_TOKENS = 256
EVAL_GENERATION_LIMIT = None  # Set an int like 64 for quick smoke tests.
DO_SAMPLE = False
TEMPERATURE = 0.0

# =========================
# Run naming
# =========================
run_timestamp = datetime.utcnow().strftime("%Y%m%d_%H%M%S")
RUN_NAME = f"factcheck_{TRAIN_MODE}_{run_timestamp}"

RUN_DIR = RUNS_DIR / RUN_NAME
CHECKPOINT_DIR = RUN_DIR / "checkpoints"
ARTIFACT_DIR = MODELS_DIR / RUN_NAME
ADAPTER_DIR = ARTIFACT_DIR / "adapter"
MERGED_DIR = ARTIFACT_DIR / "merged"
EVAL_DIR = REPORTS_DIR / RUN_NAME

for folder in [RUN_DIR, CHECKPOINT_DIR, ARTIFACT_DIR, ADAPTER_DIR, MERGED_DIR, EVAL_DIR]:
    folder.mkdir(parents=True, exist_ok=True)


CONFIG = {
    "train_mode": TRAIN_MODE,
    "base_model": BASE_MODEL,
    "seed": SEED,
    "val_size": VAL_SIZE,
    "test_size": TEST_SIZE,
    "max_length_cap": MAX_LENGTH_CAP,
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "lora_dropout": LORA_DROPOUT,
    "lora_target_modules": LORA_TARGET_MODULES,
    "per_device_train_batch_size": PER_DEVICE_TRAIN_BATCH_SIZE,
    "per_device_eval_batch_size": PER_DEVICE_EVAL_BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "num_epochs_lora": NUM_EPOCHS_LORA,
    "num_epochs_full": NUM_EPOCHS_FULL,
    "learning_rate_lora": LEARNING_RATE_LORA,
    "learning_rate_full": LEARNING_RATE_FULL,
    "weight_decay": WEIGHT_DECAY,
    "warmup_ratio": WARMUP_RATIO,
    "lr_scheduler_type": LR_SCHEDULER_TYPE,
    "max_grad_norm": MAX_GRAD_NORM,
    "logging_steps": LOGGING_STEPS,
    "eval_steps": EVAL_STEPS,
    "save_steps": SAVE_STEPS,
    "save_total_limit": SAVE_TOTAL_LIMIT,
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "eval_max_new_tokens": EVAL_MAX_NEW_TOKENS,
    "eval_generation_limit": EVAL_GENERATION_LIMIT,
    "do_sample": DO_SAMPLE,
    "temperature": TEMPERATURE,
    "run_name": RUN_NAME,
    "run_dir": str(RUN_DIR),
    "artifact_dir": str(ARTIFACT_DIR),
}

with open(RUN_DIR / "config.json", "w", encoding="utf-8") as f:
    json.dump(CONFIG, f, indent=2)

print(json.dumps(CONFIG, indent=2))

{
  "train_mode": "lora",
  "base_model": "meta-llama/Llama-3.1-8B-Instruct",
  "seed": 3407,
  "val_size": 0.1,
  "test_size": 0.1,
  "max_length_cap": 2048,
  "lora_r": 32,
  "lora_alpha": 32,
  "lora_dropout": 0.0,
  "lora_target_modules": [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    "gate_proj",
    "up_proj",
    "down_proj"
  ],
  "per_device_train_batch_size": 1,
  "per_device_eval_batch_size": 1,
  "gradient_accumulation_steps": 8,
  "num_epochs_lora": 4,
  "num_epochs_full": 3,
  "learning_rate_lora": 0.0002,
  "learning_rate_full": 1e-05,
  "weight_decay": 0.01,
  "warmup_ratio": 0.05,
  "lr_scheduler_type": "cosine",
  "max_grad_norm": 1.0,
  "logging_steps": 10,
  "eval_steps": 50,
  "save_steps": 50,
  "save_total_limit": 2,
  "early_stopping_patience": 2,
  "eval_max_new_tokens": 256,
  "eval_generation_limit": null,
  "do_sample": false,
  "temperature": 0.0,
  "run_name": "factcheck_lora_20260315_213108",
  "run_dir": "/content/drive/MyDrive/AIGColab/NL

/tmp/ipykernel_648/3002270559.py:82: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  run_timestamp = datetime.utcnow().strftime("%Y%m%d_%H%M%S")


## 5) Imports, reproducibility, and helper utilities

In [6]:
import gc
import math
import random
import re

import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from peft import LoraConfig, PeftModel
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    EarlyStoppingCallback,
    set_seed,
)
from trl import SFTConfig, SFTTrainer

# Set all common random seeds for reproducibility.
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
set_seed(SEED)

# Use BF16 when the GPU supports it, otherwise fall back to FP16.
USE_BF16 = bool(torch.cuda.is_available() and hasattr(torch.cuda, "is_bf16_supported") and torch.cuda.is_bf16_supported())
TORCH_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16

print("USE_BF16   =", USE_BF16)
print("TORCH_DTYPE =", TORCH_DTYPE)

def clean_gpu_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def next_multiple_of(value: int, multiple: int) -> int:
    return int(math.ceil(value / multiple) * multiple)

USE_BF16   = True
TORCH_DTYPE = torch.bfloat16


## 6) Load the dataset and run basic quality checks

This notebook expects your dataset to be a JSON list with:
- `prompt`
- `completion`
- `metadata`

Your uploaded file already follows that structure.

In [7]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH}. Put training_pairs.json there and rerun."
    )

with open(DATA_PATH, "r", encoding="utf-8") as f:
    raw_rows = json.load(f)

if not isinstance(raw_rows, list):
    raise ValueError("Expected a JSON list of training rows.")

df = pd.DataFrame(raw_rows)

required_columns = {"prompt", "completion", "metadata"}
missing = required_columns - set(df.columns)
if missing:
    raise ValueError(f"Dataset is missing required columns: {missing}")

# Pull a few metadata fields up to the top level for easier analysis.
df["label"] = df["metadata"].apply(lambda x: x.get("label") if isinstance(x, dict) else None)
# df["merged_label"] = df["metadata"].apply(lambda x: x.get("merged_label") if isinstance(x, dict) else None)
# df["claim_type"] = df["metadata"].apply(lambda x: x.get("claim_type") if isinstance(x, dict) else None)
# df["claim_id"] = df["metadata"].apply(lambda x: x.get("claim_id") if isinstance(x, dict) else None)

print("Rows loaded:", len(df))
print("\nLabel distribution:")
print(df["label"].value_counts(dropna=False))
# print("\nClaim type distribution:")
# print(df["claim_type"].value_counts(dropna=False).head(20))

# Remove exact duplicate prompt/completion pairs.
before = len(df)
df = df.drop_duplicates(subset=["prompt", "completion"]).reset_index(drop=True)
after = len(df)

print(f"\nRemoved exact duplicate rows: {before - after}")
print("Rows remaining:", after)

# Save a cleaned copy for reproducibility.
cleaned_data_path = RUN_DIR / "training_pairs.cleaned.json"
df.to_json(cleaned_data_path, orient="records", force_ascii=False, indent=2)
print("Saved cleaned dataset copy to:", cleaned_data_path)

# display(df[["label", "merged_label", "claim_type", "prompt", "completion"]].head(3))
display(df[["label", "prompt", "completion"]].head(3))


Rows loaded: 1289

Label distribution:
label
TRUE          500
FALSE         500
MISLEADING    248
UNVERIFIED     41
Name: count, dtype: int64

Removed exact duplicate rows: 0
Rows remaining: 1289
Saved cleaned dataset copy to: /content/drive/MyDrive/AIGColab/NLP/FinalProject/runs/factcheck_lora_20260315_213108/training_pairs.cleaned.json


,label,prompt,completion
0,TRUE,Analyze this claim from Canadian Parliament an...,VERDICT: TRUE\nCONFIDENCE: 95\nEXPLANATION: St...
1,MISLEADING,Analyze this claim from Canadian Parliament an...,VERDICT: MISLEADING\nCONFIDENCE: 90\nEXPLANATI...
2,MISLEADING,Analyze this claim from Canadian Parliament an...,VERDICT: MISLEADING\nCONFIDENCE: 80\nEXPLANATI...


## 7) Train / validation / test split

Best practice: keep a real test set that you never train on.
We stratify by the final label so each split keeps roughly the same class balance.

In [8]:
if df["label"].isna().any():
    raise ValueError("Some rows are missing metadata.label. Fix the dataset before training.")

train_df, temp_df = train_test_split(
    df,
    test_size=VAL_SIZE + TEST_SIZE,
    random_state=SEED,
    stratify=df["label"],
)

relative_test_size = TEST_SIZE / (VAL_SIZE + TEST_SIZE)

val_df, test_df = train_test_split(
    temp_df,
    test_size=relative_test_size,
    random_state=SEED,
    stratify=temp_df["label"],
)

print("Train size:", len(train_df))
print("Val size:  ", len(val_df))
print("Test size: ", len(test_df))

print("\nTrain label distribution:")
print(train_df["label"].value_counts(normalize=True).round(3))

print("\nVal label distribution:")
print(val_df["label"].value_counts(normalize=True).round(3))

print("\nTest label distribution:")
print(test_df["label"].value_counts(normalize=True).round(3))

# Persist the exact splits so future reruns use the same samples if needed.
train_split_path = RUN_DIR / "train_split.json"
val_split_path = RUN_DIR / "val_split.json"
test_split_path = RUN_DIR / "test_split.json"

train_df.to_json(train_split_path, orient="records", force_ascii=False, indent=2)
val_df.to_json(val_split_path, orient="records", force_ascii=False, indent=2)
test_df.to_json(test_split_path, orient="records", force_ascii=False, indent=2)

print("\nSaved split files.")

Train size: 1031
Val size:   129
Test size:  129

Train label distribution:
label
FALSE         0.388
TRUE          0.388
MISLEADING    0.192
UNVERIFIED    0.032
Name: proportion, dtype: float64

Val label distribution:
label
FALSE         0.388
TRUE          0.388
MISLEADING    0.194
UNVERIFIED    0.031
Name: proportion, dtype: float64

Test label distribution:
label
TRUE          0.388
FALSE         0.388
MISLEADING    0.194
UNVERIFIED    0.031
Name: proportion, dtype: float64

Saved split files.


## 8) Build Hugging Face datasets

We keep the dataset in **prompt/completion** format because it already matches your project task cleanly.
That avoids adding extra chat-template behavior you do not need.

In [9]:
# Keep only the columns we actually need for training and evaluation.
# kept_columns = ["prompt", "completion", "label", "merged_label", "claim_type", "claim_id"]
kept_columns = ["prompt", "completion", "label"]

train_ds = Dataset.from_pandas(train_df[kept_columns], preserve_index=False)
val_ds = Dataset.from_pandas(val_df[kept_columns], preserve_index=False)
test_ds = Dataset.from_pandas(test_df[kept_columns], preserve_index=False)

print(train_ds)
print(val_ds)
print(test_ds)

Dataset({
    features: ['prompt', 'completion', 'label'],
    num_rows: 1031
})
Dataset({
    features: ['prompt', 'completion', 'label'],
    num_rows: 129
})
Dataset({
    features: ['prompt', 'completion', 'label'],
    num_rows: 129
})


## 9) Load the tokenizer and estimate a sensible max sequence length

Do not guess blindly.
We inspect token lengths on the actual dataset and then choose a value that is safe but not wasteful.

In [10]:
tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    token=HF_TOKEN,
    use_fast=True,
)

# Llama-style causal LMs usually do not define a pad token.
# For training, using eos as pad is the standard practical fix.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

def tokenize_length(example):
    # We measure the exact sequence the trainer will learn from: prompt + completion.
    completion = example["completion"]
    if tokenizer.eos_token and not completion.endswith(tokenizer.eos_token):
        completion = completion + tokenizer.eos_token

    text = example["prompt"] + completion
    tokens = tokenizer(text, add_special_tokens=True, truncation=False)
    return {"token_length": len(tokens["input_ids"])}

length_probe = Dataset.from_pandas(df[["prompt", "completion"]], preserve_index=False)
length_probe = length_probe.map(tokenize_length)

token_lengths = length_probe["token_length"]

p50 = int(np.percentile(token_lengths, 50))
p90 = int(np.percentile(token_lengths, 90))
p95 = int(np.percentile(token_lengths, 95))
p99 = int(np.percentile(token_lengths, 99))
max_seen = int(np.max(token_lengths))

AUTO_MAX_LENGTH = next_multiple_of(min(MAX_LENGTH_CAP, max(1024, p99 + 64)), 64)

print("Token length stats")
print("p50 =", p50)
print("p90 =", p90)
print("p95 =", p95)
print("p99 =", p99)
print("max =", max_seen)
print("AUTO_MAX_LENGTH =", AUTO_MAX_LENGTH)

MAX_LENGTH = AUTO_MAX_LENGTH

# Count how many examples would be truncated at the chosen max length.
truncated_count = int(sum(length > MAX_LENGTH for length in token_lengths))
print(f"Rows longer than MAX_LENGTH ({MAX_LENGTH}):", truncated_count)

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Map:   0%|          | 0/1289 [00:00<?, ? examples/s]

Token length stats
p50 = 356
p90 = 414
p95 = 429
p99 = 457
max = 516
AUTO_MAX_LENGTH = 1024
Rows longer than MAX_LENGTH (1024): 0


## 10) Load the model

This is the only meaningful branch in the notebook:
- **LoRA mode**: train adapters only
- **full mode**: train all weights

Everything else stays the same.

In [11]:
clean_gpu_memory()

common_model_kwargs = {
    "token": HF_TOKEN,
    "torch_dtype": TORCH_DTYPE,
    "low_cpu_mem_usage": True,
    "trust_remote_code": False,
}

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    **common_model_kwargs,
)

# Disable cache during training so gradient checkpointing can work properly.
model.config.use_cache = False

# Enable gradient checkpointing to reduce memory pressure.
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})

peft_config = None

if TRAIN_MODE == "lora":
    # In LoRA mode, we keep the base weights frozen and train only lightweight adapters.
    model.enable_input_require_grads()

    peft_config = LoraConfig(
        task_type="CAUSAL_LM",
        inference_mode=False,
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        target_modules=LORA_TARGET_MODULES,
    )
    print("Initialized model in LoRA mode.")
elif TRAIN_MODE == "full":
    print("Initialized model in full fine-tuning mode.")
else:
    raise ValueError("TRAIN_MODE must be either 'lora' or 'full'.")

print("Pad token id:", tokenizer.pad_token_id)
print("EOS token id:", tokenizer.eos_token_id)

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

Initialized model in LoRA mode.
Pad token id: 128009
EOS token id: 128009


## 11) Training arguments

The hyperparameters are intentionally separate for LoRA and full FT.

Why:
- LoRA usually tolerates a **higher learning rate**
- full FT usually needs a **lower learning rate**

In [12]:
# Append EOS to completions so the model learns where to stop.
def add_eos(text: str) -> str:
    text = text or ""
    if tokenizer.eos_token and not text.endswith(tokenizer.eos_token):
        return text + tokenizer.eos_token
    return text

for split_df in [train_df, val_df, test_df]:
    split_df["completion"] = split_df["completion"].apply(add_eos)

# Rebuild HF datasets after EOS update.
kept_columns = ["prompt", "completion", "label"]

train_ds = Dataset.from_pandas(train_df[kept_columns], preserve_index=False)
val_ds = Dataset.from_pandas(val_df[kept_columns], preserve_index=False)
test_ds = Dataset.from_pandas(test_df[kept_columns], preserve_index=False)

print("EOS appended and datasets rebuilt.")

EOS appended and datasets rebuilt.


In [13]:
current_learning_rate = LEARNING_RATE_LORA if TRAIN_MODE == "lora" else LEARNING_RATE_FULL
current_num_epochs = NUM_EPOCHS_LORA if TRAIN_MODE == "lora" else NUM_EPOCHS_FULL

training_args = SFTConfig(
    output_dir=str(CHECKPOINT_DIR),

    # Batch and optimization
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=current_learning_rate,
    num_train_epochs=current_num_epochs,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type=LR_SCHEDULER_TYPE,
    max_grad_norm=MAX_GRAD_NORM,
    optim="adamw_8bit",

    # Precision
    bf16=USE_BF16,
    fp16=not USE_BF16,

    # Memory / speed
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},

    # Logging / evaluation / checkpointing
    logging_steps=LOGGING_STEPS,
    logging_first_step=True,
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=SAVE_TOTAL_LIMIT,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",

    # Dataset handling
    max_length=MAX_LENGTH,
    packing=False,                  # Keep it explicit and easy to debug first.
    completion_only_loss=True,      # Learn only on the target completion.
    dataset_num_proc=2,

    # Reproducibility
    seed=SEED,
    data_seed=SEED,
)

print(training_args)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


SFTConfig(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
activation_offloading=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
assistant_only_loss=False,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=True,
bf16_full_eval=False,
chat_template_path=None,
completion_only_loss=True,
data_seed=3407,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
dataset_kwargs=None,
dataset_num_proc=2,
dataset_text_field=text,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
enable_jit_checkpoint=False,
eos_token=<

## 12) Build the trainer

In [14]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    peft_config=peft_config,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=EARLY_STOPPING_PATIENCE
        )
    ],
)

# Quick sanity check: show trainable parameter count.
def print_trainable_parameters(model):
    trainable_params = 0
    all_params = 0
    for _, param in model.named_parameters():
        all_params += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()
    pct = 100 * trainable_params / all_params
    print(f"Trainable params: {trainable_params:,}")
    print(f"All params:       {all_params:,}")
    print(f"Trainable %:      {pct:.4f}%")

print_trainable_parameters(trainer.model)

Adding EOS to train dataset (num_proc=2):   0%|          | 0/1031 [00:00<?, ? examples/s]

Tokenizing train dataset (num_proc=2):   0%|          | 0/1031 [00:00<?, ? examples/s]

Truncating train dataset (num_proc=2):   0%|          | 0/1031 [00:00<?, ? examples/s]

Adding EOS to eval dataset (num_proc=2):   0%|          | 0/129 [00:00<?, ? examples/s]

Tokenizing eval dataset (num_proc=2):   0%|          | 0/129 [00:00<?, ? examples/s]

Truncating eval dataset (num_proc=2):   0%|          | 0/129 [00:00<?, ? examples/s]

Trainable params: 83,886,080
All params:       8,114,147,328
Trainable %:      1.0338%


## 13) Train

This is the actual fine-tuning step.

In [15]:
gpu_stats = None
if torch.cuda.is_available():
    gpu_stats = torch.cuda.get_device_properties(0)
    start_reserved_gb = torch.cuda.max_memory_reserved() / 1024**3
    print(f"GPU: {gpu_stats.name}")
    print(f"Total VRAM: {gpu_stats.total_memory / 1024**3:.2f} GB")
    print(f"Reserved before training: {start_reserved_gb:.2f} GB")

train_result = trainer.train()
trainer.save_state()

metrics = train_result.metrics
metrics["best_model_checkpoint"] = trainer.state.best_model_checkpoint
metrics["train_mode"] = TRAIN_MODE
metrics["max_length"] = MAX_LENGTH

with open(RUN_DIR / "train_metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

print(json.dumps(metrics, indent=2))

if torch.cuda.is_available():
    peak_reserved_gb = torch.cuda.max_memory_reserved() / 1024**3
    print(f"Peak reserved memory: {peak_reserved_gb:.2f} GB")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.


GPU: NVIDIA A100-SXM4-80GB
Total VRAM: 79.25 GB
Reserved before training: 15.31 GB


Step,Training Loss,Validation Loss
50,0.400185,0.345572
100,0.278652,0.313941
150,0.206926,0.303934
200,0.206060,0.299148
250,0.177815,0.285223
300,0.120952,0.306516
350,0.113452,0.310713


{
  "train_runtime": 1474.1492,
  "train_samples_per_second": 2.798,
  "train_steps_per_second": 0.35,
  "total_flos": 4.448269452489523e+16,
  "train_loss": 0.2749818820612771,
  "best_model_checkpoint": "/content/drive/MyDrive/AIGColab/NLP/FinalProject/runs/factcheck_lora_20260315_213108/checkpoints/checkpoint-250",
  "train_mode": "lora",
  "max_length": 1024
}
Peak reserved memory: 20.39 GB


## 14) Save the trained model artifacts

Behavior:
- **LoRA mode** → save adapter weights first
- **full mode** → save the full trained model directly

In [16]:
if TRAIN_MODE == "lora":
    trainer.model.save_pretrained(ADAPTER_DIR)
    tokenizer.save_pretrained(ADAPTER_DIR)
    print("Saved LoRA adapter to:", ADAPTER_DIR)
else:
    trainer.model.save_pretrained(MERGED_DIR, safe_serialization=True)
    tokenizer.save_pretrained(MERGED_DIR)
    print("Saved full fine-tuned model to:", MERGED_DIR)

Saved LoRA adapter to: /content/drive/MyDrive/AIGColab/NLP/FinalProject/models/factcheck_lora_20260315_213108/adapter


## 15) Merge the LoRA adapter into a standalone model

This step is **only needed in LoRA mode**.
After this step, you have a normal Hugging Face model folder that can be converted to GGUF.

In [17]:
if TRAIN_MODE == "lora":
    clean_gpu_memory()

    # Reload the base model cleanly, then attach the adapter and merge it.
    base_model_for_merge = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        token=HF_TOKEN,
        torch_dtype=TORCH_DTYPE,
        low_cpu_mem_usage=True,
        trust_remote_code=False,
    )

    merged_model = PeftModel.from_pretrained(
        base_model_for_merge,
        ADAPTER_DIR,
    )

    merged_model = merged_model.merge_and_unload()
    merged_model.save_pretrained(
        MERGED_DIR,
        safe_serialization=True,
        max_shard_size="5GB",
    )
    tokenizer.save_pretrained(MERGED_DIR)

    print("Merged LoRA adapter into standalone model:", MERGED_DIR)

    # Clean temporary merge objects.
    del base_model_for_merge
    del merged_model
    clean_gpu_memory()
else:
    print("Skipping merge because TRAIN_MODE='full'.")

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/4 [00:00<?, ?it/s]

Merged LoRA adapter into standalone model: /content/drive/MyDrive/AIGColab/NLP/FinalProject/models/factcheck_lora_20260315_213108/merged


## 16) Generation-based evaluation on the held-out test set

Loss alone is not enough for this project.
We also evaluate:
- verdict accuracy
- macro F1
- exact output-format compliance
- whether the model actually emits one of your allowed verdict labels

This is much closer to the real API task.

In [18]:
# For evaluation, the currently trained model in memory is still usable.
# We do not need the merged model for testing quality right now.

VERDICT_PATTERN = re.compile(r"(?im)^\s*VERDICT:\s*(TRUE|FALSE|MISLEADING|UNVERIFIED)\s*$")
CONFIDENCE_PATTERN = re.compile(r"(?im)^\s*CONFIDENCE:\s*([0-9]{1,3})\s*$")
EXPLANATION_PATTERN = re.compile(r"(?im)^\s*EXPLANATION:\s*(.+)$")
CORRECTION_PATTERN = re.compile(r"(?im)^\s*CORRECTION:\s*(.+)$")

def extract_verdict(text: str):
    if not isinstance(text, str):
        return None
    match = VERDICT_PATTERN.search(text)
    return match.group(1) if match else None

def has_required_format(text: str) -> bool:
    if not isinstance(text, str):
        return False
    return all([
        VERDICT_PATTERN.search(text) is not None,
        CONFIDENCE_PATTERN.search(text) is not None,
        EXPLANATION_PATTERN.search(text) is not None,
        CORRECTION_PATTERN.search(text) is not None,
    ])

def generate_completion(prompt: str) -> str:
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH,
    )

    inputs = {k: v.to(trainer.model.device) for k, v in inputs.items()}

    generation_kwargs = dict(
        max_new_tokens=EVAL_MAX_NEW_TOKENS,
        do_sample=DO_SAMPLE,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    if DO_SAMPLE:
        generation_kwargs["temperature"] = TEMPERATURE

    with torch.no_grad():
        generated = trainer.model.generate(
            **inputs,
            **generation_kwargs,
        )

    # We want only the newly generated continuation, not the prompt itself.
    new_tokens = generated[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

test_rows = list(test_ds)
if EVAL_GENERATION_LIMIT is not None:
    test_rows = test_rows[:EVAL_GENERATION_LIMIT]

pred_records = []
for idx, row in enumerate(test_rows, start=1):
    prediction_text = generate_completion(row["prompt"])
    pred_records.append({
        "row_index": idx,
        "claim_id": row.get("claim_id"),
        "claim_type": row.get("claim_type"),
        "gold_label": row.get("label"),
        "prompt": row["prompt"],
        "gold_completion": row["completion"],
        "pred_completion": prediction_text,
        "gold_verdict": row.get("label"),
        "pred_verdict": extract_verdict(prediction_text),
        "format_ok": has_required_format(prediction_text),
    })

    if idx % 10 == 0:
        print(f"Generated {idx}/{len(test_rows)} test predictions")

pred_df = pd.DataFrame(pred_records)

# Some predictions may fail to emit a valid verdict. We keep that visible.
valid_mask = pred_df["pred_verdict"].notna()

metrics_summary = {
    "evaluated_rows": int(len(pred_df)),
    "rows_with_valid_verdict": int(valid_mask.sum()),
    "valid_verdict_rate": float(valid_mask.mean()) if len(pred_df) else 0.0,
    "format_compliance_rate": float(pred_df["format_ok"].mean()) if len(pred_df) else 0.0,
}

if valid_mask.any():
    y_true = pred_df.loc[valid_mask, "gold_verdict"]
    y_pred = pred_df.loc[valid_mask, "pred_verdict"]

    metrics_summary["verdict_accuracy"] = float(accuracy_score(y_true, y_pred))
    metrics_summary["macro_f1"] = float(f1_score(y_true, y_pred, average="macro"))

    class_report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    confusion = confusion_matrix(
        y_true,
        y_pred,
        labels=["TRUE", "FALSE", "MISLEADING", "UNVERIFIED"],
    )
else:
    class_report = {}
    confusion = []

pred_path = EVAL_DIR / "test_predictions.csv"
summary_path = EVAL_DIR / "test_metrics.json"
report_path = EVAL_DIR / "classification_report.json"
confusion_path = EVAL_DIR / "confusion_matrix.json"

pred_df.to_csv(pred_path, index=False)
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(metrics_summary, f, indent=2)
with open(report_path, "w", encoding="utf-8") as f:
    json.dump(class_report, f, indent=2)
with open(confusion_path, "w", encoding="utf-8") as f:
    json.dump(confusion.tolist() if hasattr(confusion, "tolist") else confusion, f, indent=2)

print("Evaluation summary:")
print(json.dumps(metrics_summary, indent=2))

display(pred_df[["gold_verdict", "pred_verdict", "format_ok", "claim_type"]].head(10))

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Generated 10/129 test predictions
Generated 20/129 test predictions
Generated 30/129 test predictions
Generated 40/129 test predictions
Generated 50/129 test predictions
Generated 60/129 test predictions
Generated 70/129 test predictions
Generated 80/129 test predictions
Generated 90/129 test predictions
Generated 100/129 test predictions
Generated 110/129 test predictions
Generated 120/129 test predictions
Evaluation summary:
{
  "evaluated_rows": 129,
  "rows_with_valid_verdict": 129,
  "valid_verdict_rate": 1.0,
  "format_compliance_rate": 1.0,
  "verdict_accuracy": 0.5968992248062015,
  "macro_f1": 0.7111037234042554
}


,gold_verdict,pred_verdict,format_ok,claim_type
0,TRUE,FALSE,True,None
1,MISLEADING,MISLEADING,True,None
2,FALSE,TRUE,True,None
3,MISLEADING,MISLEADING,True,None
4,MISLEADING,MISLEADING,True,None
5,FALSE,FALSE,True,None
6,MISLEADING,MISLEADING,True,None
7,TRUE,TRUE,True,None
8,FALSE,TRUE,True,None
9,TRUE,TRUE,True,None


## 17) Quick manual sanity check

Before exporting, inspect a few examples manually.

In [19]:
sample_n = min(3, len(pred_df))
for i in range(sample_n):
    row = pred_df.iloc[i]
    print("=" * 120)
    print(f"Example {i+1}")
    print("- Prompt:")
    print(row["prompt"][:1200], "..." if len(row["prompt"]) > 1200 else "")
    print("\n- Gold completion:")
    print(row["gold_completion"])
    print("\n- Predicted completion:")
    print(row["pred_completion"])
    print("\n- Gold verdict:", row["gold_verdict"])
    print("- Pred verdict :", row["pred_verdict"])
    print("- Format OK    :", row["format_ok"])

Example 1
- Prompt:
Analyze this claim from Canadian Parliament and provide a fact-check:

Claim: "On Monday, we will begin second reading debate of Bill C-63 , the online harms act."
Context: and wonderful to be back. I want to wish members a good return. I hope everybody had a productive and happy time with their families and their constituents in their ridings. This afternoon, we will resume second reading debate of Bill C-66 , the military justice system modernization act. Tomorrow, we will begin the report stage debate of Bill C-33 , the strengthening the port system and railway safety in Canada act. On Monday, we will begin second reading debate of Bill C-63 , the online harms act. Madam Speaker, you will be very happy to know that next Wednesday we will also be resuming second reading debate of Bill C-71 , which would amend the Citizenship Act. I would also like to take the opportunity to inform the House that both next Tuesday and next Thursday shall be allotted days. Furthermo

## 18) GGUF export moved to a separate notebook

To avoid overloading the Colab training runtime, GGUF conversion now lives in `notebooks/factcheck_gguf_export_colab.ipynb`.
Run that notebook later, using the `MERGED_DIR` produced here as input.


In [20]:
print("GGUF export is intentionally not run in this notebook.")
print("Use notebooks/factcheck_gguf_export_colab.ipynb after training finishes.")
print("Merged HF model directory:", MERGED_DIR)


GGUF export is intentionally not run in this notebook.
Use notebooks/factcheck_gguf_export_colab.ipynb after training finishes.
Merged HF model directory: /content/drive/MyDrive/AIGColab/NLP/FinalProject/models/factcheck_lora_20260315_213108/merged


## 19) Final artifact summary

This summary now reports only the artifacts produced by the training/evaluation notebook.


In [21]:
artifact_summary = {
    "run_dir": str(RUN_DIR),
    "adapter_dir": str(ADAPTER_DIR) if TRAIN_MODE == "lora" else None,
    "merged_model_dir": str(MERGED_DIR),
    "evaluation_dir": str(EVAL_DIR),
    "predictions_csv": str(EVAL_DIR / "test_predictions.csv"),
    "metrics_json": str(EVAL_DIR / "test_metrics.json"),
}

with open(RUN_DIR / "artifact_summary.json", "w", encoding="utf-8") as f:
    json.dump(artifact_summary, f, indent=2)

print(json.dumps(artifact_summary, indent=2))


{
  "run_dir": "/content/drive/MyDrive/AIGColab/NLP/FinalProject/runs/factcheck_lora_20260315_213108",
  "adapter_dir": "/content/drive/MyDrive/AIGColab/NLP/FinalProject/models/factcheck_lora_20260315_213108/adapter",
  "merged_model_dir": "/content/drive/MyDrive/AIGColab/NLP/FinalProject/models/factcheck_lora_20260315_213108/merged",
  "evaluation_dir": "/content/drive/MyDrive/AIGColab/NLP/FinalProject/reports/factcheck_lora_20260315_213108",
  "predictions_csv": "/content/drive/MyDrive/AIGColab/NLP/FinalProject/reports/factcheck_lora_20260315_213108/test_predictions.csv",
  "metrics_json": "/content/drive/MyDrive/AIGColab/NLP/FinalProject/reports/factcheck_lora_20260315_213108/test_metrics.json"
}


## 20) Optional: minimal inference helper for later use

Use this after training, or in a new notebook after loading the merged model.

In [22]:
def fact_check_claim(prompt: str, model, tokenizer, max_new_tokens: int = 256) -> str:
    """
    Generate a fact-check completion from a raw project prompt.
    """
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH,
    )

    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    continuation = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(continuation, skip_special_tokens=True).strip()

# Example:
sample_prompt = test_ds[0]["prompt"]
print(fact_check_claim(sample_prompt, trainer.model, tokenizer))

VERDICT: FALSE
CONFIDENCE: 100
EXPLANATION: False. The bill status record shows Bill C-66 had status 'was at second reading in the house' on 2025-10-03, not 'was at second reading in the senate'.
CORRECTION: The bill status record shows Bill C-66 had status 'was at second reading in the house' on 2025-10-03.


## 21) Free GPU memory before comparison

In [23]:
# Free the training objects from GPU memory before loading fresh models for comparison.
# This avoids holding the trainer model + base model + merged model at the same time.

def clean_gpu_memory():
    import gc
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

if "trainer" in globals():
    try:
        trainer.model.cpu()
    except Exception:
        pass
    del trainer

if "model" in globals():
    try:
        model.cpu()
    except Exception:
        pass
    del model

clean_gpu_memory()
print("Freed training model objects from memory.")

Freed training model objects from memory.


## 22) Load the exact held-out test split and define reusable evaluation helpers

In [24]:
from pathlib import Path
import gc
import json
import re

import numpy as np
import pandas as pd
import torch

from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from transformers import AutoModelForCausalLM, AutoTokenizer

# -------------------------------------------------------------------
# Comparison settings
# -------------------------------------------------------------------
# Keep these the same for both base and fine-tuned models.
COMPARE_EVAL_LIMIT = None   # Example: set to 64 for a quick smoke test
COMPARE_BATCH_PROGRESS_EVERY = 10

# Model paths:
# - BASE_MODEL already exists in the notebook config
# - MERGED_DIR already exists in the notebook config and points to:
#     * merged LoRA model, or
#     * full fine-tuned model directory
BASELINE_MODEL_REF = BASE_MODEL
FINETUNED_MODEL_REF = str(MERGED_DIR)

# Exact saved test split from your earlier split cell.
TEST_SPLIT_PATH = RUN_DIR / "test_split.json"

if TEST_SPLIT_PATH.exists():
    test_eval_df = pd.read_json(TEST_SPLIT_PATH)
    print(f"Loaded saved test split from: {TEST_SPLIT_PATH}")
elif "test_df" in globals():
    test_eval_df = test_df.copy()
    print("Using test_df still present in memory.")
else:
    raise FileNotFoundError(
        "Could not find the held-out test split. Expected RUN_DIR / 'test_split.json'."
    )

required_cols = ["prompt", "completion", "label"]
missing_cols = [col for col in required_cols if col not in test_eval_df.columns]
if missing_cols:
    raise ValueError(f"Test split is missing required columns: {missing_cols}")

if COMPARE_EVAL_LIMIT is not None:
    test_eval_df = test_eval_df.head(COMPARE_EVAL_LIMIT).copy()

print("Rows to evaluate:", len(test_eval_df))
print("\nGold label distribution:")
print(test_eval_df["label"].value_counts(dropna=False))

# -------------------------------------------------------------------
# Parsing helpers
# -------------------------------------------------------------------
EVAL_LABELS = ["TRUE", "FALSE", "MISLEADING", "UNVERIFIED"]

VERDICT_PATTERN = re.compile(r"(?im)^\s*VERDICT:\s*(TRUE|FALSE|MISLEADING|UNVERIFIED)\s*$")
CONFIDENCE_PATTERN = re.compile(r"(?im)^\s*CONFIDENCE:\s*([0-9]{1,3})\s*$")
EXPLANATION_PATTERN = re.compile(r"(?im)^\s*EXPLANATION:\s*(.+)$")
CORRECTION_PATTERN = re.compile(r"(?im)^\s*CORRECTION:\s*(.+)$")

def extract_verdict(text: str):
    if not isinstance(text, str):
        return None
    match = VERDICT_PATTERN.search(text)
    return match.group(1) if match else None

def has_required_format(text: str) -> bool:
    if not isinstance(text, str):
        return False
    return all([
        VERDICT_PATTERN.search(text) is not None,
        CONFIDENCE_PATTERN.search(text) is not None,
        EXPLANATION_PATTERN.search(text) is not None,
        CORRECTION_PATTERN.search(text) is not None,
    ])

def build_generation_kwargs(tokenizer):
    generation_kwargs = {
        "max_new_tokens": EVAL_MAX_NEW_TOKENS,
        "do_sample": DO_SAMPLE,
        "pad_token_id": tokenizer.pad_token_id,
        "eos_token_id": tokenizer.eos_token_id,
    }
    if DO_SAMPLE:
        generation_kwargs["temperature"] = TEMPERATURE
    return generation_kwargs

def load_eval_model(model_ref: str):
    """
    Load a causal LM + tokenizer for evaluation only.
    """
    eval_dtype = TORCH_DTYPE if torch.cuda.is_available() else torch.float32

    tokenizer = AutoTokenizer.from_pretrained(
        model_ref,
        token=HF_TOKEN,
        use_fast=True,
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    model = AutoModelForCausalLM.from_pretrained(
        model_ref,
        token=HF_TOKEN,
        torch_dtype=eval_dtype,
        low_cpu_mem_usage=True,
        trust_remote_code=False,
    )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()

    return model, tokenizer, device

def generate_completion_with_model(prompt: str, model, tokenizer, device) -> str:
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH,
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            **build_generation_kwargs(tokenizer),
        )

    continuation = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(continuation, skip_special_tokens=True).strip()

def evaluate_model_on_test_split(model_ref: str, model_tag: str, eval_df: pd.DataFrame):
    """
    Run generation-based evaluation on a given model against the exact same held-out test split.
    """
    clean_gpu_memory()
    model, tokenizer, device = load_eval_model(model_ref)

    pred_records = []
    total_rows = len(eval_df)

    for idx, row in enumerate(eval_df.itertuples(index=False), start=1):
        pred_text = generate_completion_with_model(row.prompt, model, tokenizer, device)

        pred_records.append({
            "row_index": idx,
            "gold_label": row.label,
            "prompt": row.prompt,
            "gold_completion": row.completion,
            "pred_completion": pred_text,
            "pred_verdict": extract_verdict(pred_text),
            "format_ok": has_required_format(pred_text),
            "model_tag": model_tag,
            "model_ref": model_ref,
        })

        if idx % COMPARE_BATCH_PROGRESS_EVERY == 0:
            print(f"[{model_tag}] Generated {idx}/{total_rows}")

    pred_df = pd.DataFrame(pred_records)

    valid_mask = pred_df["pred_verdict"].notna()

    summary = {
        "model_tag": model_tag,
        "model_ref": model_ref,
        "evaluated_rows": int(len(pred_df)),
        "rows_with_valid_verdict": int(valid_mask.sum()),
        "valid_verdict_rate": float(valid_mask.mean()) if len(pred_df) else 0.0,
        "format_compliance_rate": float(pred_df["format_ok"].mean()) if len(pred_df) else 0.0,
    }

    if valid_mask.any():
        y_true = pred_df.loc[valid_mask, "gold_label"]
        y_pred = pred_df.loc[valid_mask, "pred_verdict"]

        summary["verdict_accuracy"] = float(accuracy_score(y_true, y_pred))
        summary["macro_f1"] = float(f1_score(y_true, y_pred, average="macro"))

        class_report = classification_report(
            y_true,
            y_pred,
            labels=EVAL_LABELS,
            output_dict=True,
            zero_division=0,
        )
        confusion = confusion_matrix(
            y_true,
            y_pred,
            labels=EVAL_LABELS,
        ).tolist()
    else:
        class_report = {}
        confusion = []

    # Clean up the model before returning.
    try:
        model.cpu()
    except Exception:
        pass

    del model
    del tokenizer
    clean_gpu_memory()

    return pred_df, summary, class_report, confusion

Loaded saved test split from: /content/drive/MyDrive/AIGColab/NLP/FinalProject/runs/factcheck_lora_20260315_213108/test_split.json
Rows to evaluate: 129

Gold label distribution:
label
TRUE          50
FALSE         50
MISLEADING    25
UNVERIFIED     4
Name: count, dtype: int64


## 23) Evaluate the untouched base model

In [25]:
base_pred_df, base_summary, base_report, base_confusion = evaluate_model_on_test_split(
    model_ref=BASELINE_MODEL_REF,
    model_tag="base_model",
    eval_df=test_eval_df,
)

print(json.dumps(base_summary, indent=2))

base_pred_path = EVAL_DIR / "baseline_test_predictions.csv"
base_summary_path = EVAL_DIR / "baseline_test_metrics.json"
base_report_path = EVAL_DIR / "baseline_classification_report.json"
base_confusion_path = EVAL_DIR / "baseline_confusion_matrix.json"

base_pred_df.to_csv(base_pred_path, index=False)

with open(base_summary_path, "w", encoding="utf-8") as f:
    json.dump(base_summary, f, indent=2)

with open(base_report_path, "w", encoding="utf-8") as f:
    json.dump(base_report, f, indent=2)

with open(base_confusion_path, "w", encoding="utf-8") as f:
    json.dump(base_confusion, f, indent=2)

print(f"Saved base-model evaluation to: {EVAL_DIR}")

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

[base_model] Generated 10/129
[base_model] Generated 20/129
[base_model] Generated 30/129
[base_model] Generated 40/129
[base_model] Generated 50/129
[base_model] Generated 60/129
[base_model] Generated 70/129
[base_model] Generated 80/129
[base_model] Generated 90/129
[base_model] Generated 100/129
[base_model] Generated 110/129
[base_model] Generated 120/129
{
  "model_tag": "base_model",
  "model_ref": "meta-llama/Llama-3.1-8B-Instruct",
  "evaluated_rows": 129,
  "rows_with_valid_verdict": 129,
  "valid_verdict_rate": 1.0,
  "format_compliance_rate": 1.0,
  "verdict_accuracy": 0.4263565891472868,
  "macro_f1": 0.341926649202191
}
Saved base-model evaluation to: /content/drive/MyDrive/AIGColab/NLP/FinalProject/reports/factcheck_lora_20260315_213108


## 24) Evaluate the fine-tuned model and build a side-by-side comparison

In [26]:
ft_pred_df, ft_summary, ft_report, ft_confusion = evaluate_model_on_test_split(
    model_ref=FINETUNED_MODEL_REF,
    model_tag="fine_tuned_model",
    eval_df=test_eval_df,
)

print(json.dumps(ft_summary, indent=2))

ft_pred_path = EVAL_DIR / "finetuned_test_predictions.csv"
ft_summary_path = EVAL_DIR / "finetuned_test_metrics.json"
ft_report_path = EVAL_DIR / "finetuned_classification_report.json"
ft_confusion_path = EVAL_DIR / "finetuned_confusion_matrix.json"

ft_pred_df.to_csv(ft_pred_path, index=False)

with open(ft_summary_path, "w", encoding="utf-8") as f:
    json.dump(ft_summary, f, indent=2)

with open(ft_report_path, "w", encoding="utf-8") as f:
    json.dump(ft_report, f, indent=2)

with open(ft_confusion_path, "w", encoding="utf-8") as f:
    json.dump(ft_confusion, f, indent=2)

# ------------------------------------------------------------
# Side-by-side comparison table
# ------------------------------------------------------------
comparison_rows = [
    {
        "metric": "evaluated_rows",
        "base_model": base_summary.get("evaluated_rows"),
        "fine_tuned_model": ft_summary.get("evaluated_rows"),
    },
    {
        "metric": "rows_with_valid_verdict",
        "base_model": base_summary.get("rows_with_valid_verdict"),
        "fine_tuned_model": ft_summary.get("rows_with_valid_verdict"),
    },
    {
        "metric": "valid_verdict_rate",
        "base_model": base_summary.get("valid_verdict_rate"),
        "fine_tuned_model": ft_summary.get("valid_verdict_rate"),
    },
    {
        "metric": "format_compliance_rate",
        "base_model": base_summary.get("format_compliance_rate"),
        "fine_tuned_model": ft_summary.get("format_compliance_rate"),
    },
    {
        "metric": "verdict_accuracy",
        "base_model": base_summary.get("verdict_accuracy"),
        "fine_tuned_model": ft_summary.get("verdict_accuracy"),
    },
    {
        "metric": "macro_f1",
        "base_model": base_summary.get("macro_f1"),
        "fine_tuned_model": ft_summary.get("macro_f1"),
    },
]

comparison_df = pd.DataFrame(comparison_rows)
comparison_df["delta_finetuned_minus_base"] = comparison_df["fine_tuned_model"] - comparison_df["base_model"]

comparison_csv_path = EVAL_DIR / "baseline_vs_finetuned_comparison.csv"
comparison_json_path = EVAL_DIR / "baseline_vs_finetuned_comparison.json"

comparison_df.to_csv(comparison_csv_path, index=False)

with open(comparison_json_path, "w", encoding="utf-8") as f:
    json.dump(comparison_df.to_dict(orient="records"), f, indent=2)

print("\nBaseline vs fine-tuned comparison:")
display(comparison_df)

print(f"\nSaved comparison files to: {EVAL_DIR}")

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

[fine_tuned_model] Generated 10/129
[fine_tuned_model] Generated 20/129
[fine_tuned_model] Generated 30/129
[fine_tuned_model] Generated 40/129
[fine_tuned_model] Generated 50/129
[fine_tuned_model] Generated 60/129
[fine_tuned_model] Generated 70/129
[fine_tuned_model] Generated 80/129
[fine_tuned_model] Generated 90/129
[fine_tuned_model] Generated 100/129
[fine_tuned_model] Generated 110/129
[fine_tuned_model] Generated 120/129
{
  "model_tag": "fine_tuned_model",
  "model_ref": "/content/drive/MyDrive/AIGColab/NLP/FinalProject/models/factcheck_lora_20260315_213108/merged",
  "evaluated_rows": 129,
  "rows_with_valid_verdict": 129,
  "valid_verdict_rate": 1.0,
  "format_compliance_rate": 1.0,
  "verdict_accuracy": 0.5813953488372093,
  "macro_f1": 0.6983670809391809
}

Baseline vs fine-tuned comparison:


,metric,base_model,fine_tuned_model,delta_finetuned_minus_base
0,evaluated_rows,129.000000,129.000000,0.000000
1,rows_with_valid_verdict,129.000000,129.000000,0.000000
2,valid_verdict_rate,1.000000,1.000000,0.000000
3,format_compliance_rate,1.000000,1.000000,0.000000
4,verdict_accuracy,0.426357,0.581395,0.155039
5,macro_f1,0.341927,0.698367,0.356440



Saved comparison files to: /content/drive/MyDrive/AIGColab/NLP/FinalProject/reports/factcheck_lora_20260315_213108
